# B7: Text Classification using BERT

Load a pre-trained BERT model with a transformer library, preprocess text with tokenization and encoding, fine-tune BERT for sentiment classification, and evaluate with accuracy, precision, recall and F1-score.

**Dataset:** IMDB Movie Reviews (a sample, so it fits a free Colab GPU) and SST-2 (Stanford Sentiment Treebank, sentence level)

**Runtime -> Change runtime type -> T4 GPU** before running. Cells that download BERT have no saved output in this file: run them in Colab.

In [2]:
# transformers : pretrained BERT, tokenizer, Trainer
# accelerate   : needed by the Trainer
# scikit-learn : metrics and the TF-IDF baseline
!pip install -q transformers accelerate scikit-learn pandas

In [3]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score, precision_recall_fscore_support

In [ ]:
print("GPU:", torch.cuda.is_available())

## Data

In [5]:
imdb = pd.read_csv("https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv")
imdb["text"] = imdb["review"].str.replace("<br />", " ")
imdb["label"] = (imdb["sentiment"] == "positive").astype(int)        # 1 = positive, 0 = negative
imdb = imdb.sample(7000, random_state=42)
imdb_train, imdb_test = train_test_split(imdb, test_size=2000, random_state=42, stratify=imdb["label"])
imdb_train, imdb_val = train_test_split(imdb_train, test_size=1000, random_state=42, stratify=imdb_train["label"])
len(imdb_train), len(imdb_val), len(imdb_test)

(4000, 1000, 2000)

In [6]:
SST = "https://raw.githubusercontent.com/clairett/pytorch-sentiment-classification/master/data/SST2/"
read = lambda part: pd.read_csv(SST + part + ".tsv", sep="\t", header=None, names=["text", "label"], quoting=3)
sst_train, sst_val, sst_test = read("train"), read("dev"), read("test")
print(len(sst_train), len(sst_val), len(sst_test))
sst_train.head(3)

6920 872 1821


,text,label
0,"a stirring , funny and finally transporting re...",1
1,apparently reassembled from the cutting room f...,0
2,they presume their audience wo n't sit still f...,0


## Baseline to beat: TF-IDF + Logistic Regression

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def baseline(train, test):
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
    model = LogisticRegression(max_iter=1000).fit(vec.fit_transform(train["text"]), train["label"])
    return accuracy_score(test["label"], model.predict(vec.transform(test["text"])))

print(f"IMDB  TF-IDF accuracy: {baseline(imdb_train, imdb_test):.3f}")
print(f"SST-2 TF-IDF accuracy: {baseline(sst_train, sst_test):.3f}")

IMDB  TF-IDF accuracy: 0.845


SST-2 TF-IDF accuracy: 0.794


## (a) Load the pre-trained BERT

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)    # adds a new 768 -> 2 layer
print(sum(p.numel() for p in model.parameters()) / 1e6, "million parameters")

## (b) Tokenization and encoding

In [ ]:
enc = tokenizer("The movie was not bad at all!")
print(tokenizer.convert_ids_to_tokens(enc["input_ids"]))
print(enc["input_ids"])
print(enc["attention_mask"])

In [ ]:
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, df, max_len):
        self.enc = tokenizer(list(df["text"]), truncation=True, padding="max_length", max_length=max_len)
        self.labels = list(df["label"])
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item

train_ds = ReviewDataset(imdb_train, 256)       # reviews are long: keep the first 256 tokens
val_ds = ReviewDataset(imdb_val, 256)
test_ds = ReviewDataset(imdb_test, 256)
train_ds[0]["input_ids"][:20]

## (c) Fine-tune

In [ ]:
from transformers import Trainer, TrainingArguments

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": accuracy_score(labels, preds), "precision": p, "recall": r, "f1": f1}

args = TrainingArguments(
    output_dir="bert-imdb",
    learning_rate=2e-5,                  # small: only nudge the pretrained weights
    per_device_train_batch_size=16,
    num_train_epochs=2,
    eval_strategy="epoch",               # score the validation set after each epoch
    save_strategy="no",
    fp16=torch.cuda.is_available(),      # faster on GPU
    report_to="none")

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)
trainer.train()

## (d) Evaluate

In [ ]:
out = trainer.predict(test_ds)
preds = out.predictions.argmax(-1)
print(classification_report(imdb_test["label"], preds, target_names=["negative", "positive"]))
ConfusionMatrixDisplay.from_predictions(imdb_test["label"], preds, display_labels=["negative", "positive"], colorbar=False)
plt.show()

In [ ]:
new = ["One of the best films of the year.", "Two hours of my life I will never get back.", "Not bad at all."]
batch = tokenizer(new, padding=True, return_tensors="pt").to(model.device)
with torch.no_grad():
    probs = torch.softmax(model(**batch).logits, dim=-1)[:, 1]
pd.DataFrame({"text": new, "P(positive)": probs.cpu().numpy().round(3)})

## SST-2: same steps

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)      # fresh copy of BERT
sst_tr, sst_va, sst_te = ReviewDataset(sst_train, 64), ReviewDataset(sst_val, 64), ReviewDataset(sst_test, 64)   # short sentences
trainer = Trainer(model=model, args=args, train_dataset=sst_tr, eval_dataset=sst_va, compute_metrics=compute_metrics)
trainer.train()
preds = trainer.predict(sst_te).predictions.argmax(-1)
print(classification_report(sst_test["label"], preds, target_names=["negative", "positive"]))

## Conclusion
- BERT is loaded with `AutoTokenizer` and `AutoModelForSequenceClassification`; the tokenizer turns text into WordPiece ids, an attention mask, and fixed-length padded inputs.
- Fine-tuning trains all weights plus the new classification layer for 2 epochs with a small learning rate.
- Compare your BERT accuracy, precision, recall and F1 with the TF-IDF baseline on the same test data: the gain is usually larger on SST-2 (short sentences where word order and negation matter) than on long IMDB reviews.
- Long reviews are cut at 256 tokens, so some text is never seen by the model.